# Sougata — BERT Base, Final Evaluation & Best-Model Analysis

**Assigned contribution:** the strongest and most presentation-focused part of the project:
BERT Base fine-tuning, final BERT test evaluation and the consolidated comparison used to identify the best-performing model.

> Shared setup cells are included only so this notebook can run independently.
> Sougata's primary contribution begins at **Part C — BERT Base**.


2. Dataset Upload

In [ ]:
from google.colab import files

uploaded = files.upload()

!pip -q install gensim datasets transformers accelerate
!unzip -o "/content/archive (4).zip" -d "/content/"

print("Environment setup completed and dataset archive extracted.")

Saving archive (4).zip to archive (4).zip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 72.8 MB/s eta 0:00:00
Archive:  /content/archive (4).zip
  inflating: /content/cyberbullying_tweets.csv  
Environment setup completed and dataset archive extracted.


## 3. Imports and Reproducibility



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import tensorflow as tf
import torch

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.probability import FreqDist
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from gensim.models import Word2Vec

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

physical_gpus = tf.config.list_physical_devices('GPU')
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("Imports completed.")
print("TensorFlow GPU devices:", physical_gpus)
print("PyTorch CUDA available:", torch.cuda.is_available())

Imports completed.
TensorFlow GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PyTorch CUDA available: True


## 4. Problem Definition

Cyberbullying is not a single homogeneous category. A content-moderation system may need to distinguish attacks based on **age, ethnicity, gender and religion**, while also separating **other cyberbullying** from **non-cyberbullying** content. This is therefore a six-class text classification problem rather than a binary toxicity or sentiment task.

The central research questions are:
1. How much do different lab-based preprocessing strategies affect validation performance?
2. How does sparse TF-IDF compare with distributed Word2Vec representations?
3. How do classical ML models compare with recurrent neural networks and BERT Base?
4. Which model provides the strongest Macro-F1 across all six classes?

## 5. Dataset Loading and Initial Inspection

The supplied CSV contains two columns: tweet text and the cyberbullying class label.

In [ ]:
DATA_PATH = "/content/cyberbullying_tweets.csv"
df = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Number of classes:", df["cyberbullying_type"].nunique())
df.head()

Raw dataset shape: (47692, 2)
Columns: ['tweet_text', 'cyberbullying_type']
Number of classes: 6


,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


## 6. Data Quality Analysis

The project requires missing-value and duplicate analysis. In addition to exact duplicate rows, this dataset contains repeated tweet texts. Some repeated texts are associated with more than one label, which would create contradictory supervision and possible leakage if the same text appears in different splits.

The cleanup policy is:
1. Remove rows with missing text or missing labels.
2. Identify tweet texts assigned to multiple different labels and remove all rows for those contradictory texts.
3. Remove remaining duplicate tweet texts with the same label.
4. Perform the train/validation/test split only after this cleanup.

In [ ]:
missing_summary = df.isna().sum()
exact_duplicate_rows = df.duplicated().sum()
unique_texts = df["tweet_text"].nunique()
repeated_text_extra_rows = len(df) - unique_texts

label_counts_per_text = df.groupby("tweet_text")["cyberbullying_type"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index
conflicting_rows = df[df["tweet_text"].isin(conflicting_texts)].shape[0]

quality_summary = pd.DataFrame({
    "Measure": [
        "Raw rows",
        "Missing tweet_text",
        "Missing cyberbullying_type",
        "Exact duplicate rows",
        "Extra rows caused by repeated tweet text",
        "Tweet texts with conflicting labels",
        "Rows involved in conflicting labels"
    ],
    "Value": [
        len(df),
        missing_summary["tweet_text"],
        missing_summary["cyberbullying_type"],
        exact_duplicate_rows,
        repeated_text_extra_rows,
        len(conflicting_texts),
        conflicting_rows
    ]
})
quality_summary

,Measure,Value
0,Raw rows,47692
1,Missing tweet_text,0
2,Missing cyberbullying_type,0
3,Exact duplicate rows,36
4,Extra rows caused by repeated tweet text,1675
5,Tweet texts with conflicting labels,1639
6,Rows involved in conflicting labels,3278


In [ ]:
df = df.dropna(subset=["tweet_text", "cyberbullying_type"]).copy()

label_counts_per_text = df.groupby("tweet_text")["cyberbullying_type"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index

df = df[~df["tweet_text"].isin(conflicting_texts)].copy()
df = df.drop_duplicates(subset=["tweet_text"]).reset_index(drop=True)

print("Dataset shape after quality cleanup:", df.shape)
print("Remaining duplicate tweet texts:", df["tweet_text"].duplicated().sum())
print("Remaining missing values:")
print(df.isna().sum())

Dataset shape after quality cleanup: (44378, 2)
Remaining duplicate tweet texts: 0
Remaining missing values:
tweet_text            0
cyberbullying_type    0
dtype: int64


## 8. Label Encoding and Stratified Train/Validation/Test Split

The cleaned dataset is split **70% / 15% / 15%**. Stratification preserves the six-class distribution in every partition and `random_state=42` makes the split reproducible.

In [ ]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["cyberbullying_type"])

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Class mapping:", dict(enumerate(label_encoder.classes_)))
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Class mapping: {0: 'age', 1: 'ethnicity', 2: 'gender', 3: 'not_cyberbullying', 4: 'other_cyberbullying', 5: 'religion'}
Train shape: (31064, 5)
Validation shape: (6657, 5)
Test shape: (6657, 5)


## 10. Shared Experiment Tracking and Evaluation Functions

Every required model is tuned with at least three explicit configurations. Validation Accuracy and validation Macro-F1 are recorded for every run. The final test evaluation records Accuracy, Macro-F1, the full classification report and a confusion matrix.

In [ ]:
tuning_records = []
final_results = []
confusion_matrices = {}
best_hyperparameters = {}


def record_tuning(model_name, representation, config_id, parameters, validation_accuracy, validation_f1):
    tuning_records.append({
        "Model": model_name,
        "Representation": representation,
        "Config": config_id,
        "Parameters": str(parameters),
        "Validation Accuracy": validation_accuracy,
        "Validation Macro F1": validation_f1
    })


def evaluate_predictions(model_name, representation, y_true, y_pred):
    test_accuracy = accuracy_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred, average="macro")
    test_confusion_matrix = confusion_matrix(y_true, y_pred)

    print("=" * 90)
    print(model_name)
    print("Representation:", representation)
    print("Test Accuracy:", round(test_accuracy, 4))
    print("Test Macro F1:", round(test_f1, 4))
    print("\nFull Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, zero_division=0))
    print("Confusion Matrix:\n", test_confusion_matrix)

    final_results.append({
        "Model": model_name,
        "Representation": representation,
        "Accuracy": test_accuracy,
        "Macro F1": test_f1
    })
    confusion_matrices[model_name] = test_confusion_matrix

    return test_accuracy, test_f1


print("Experiment tracking and evaluation functions are ready.")

Experiment tracking and evaluation functions are ready.


# Part C — BERT Base

## 23. BERT Base Fine-Tuning Strategy

The required transformer is **`bert-base-uncased`**, following Lab 4. BERT receives the original tweet text rather than aggressively stemmed or lemmatized text because its own subword tokenizer and contextual encoder are designed to operate on natural text.

To make the mandatory three-configuration tuning practical in Colab:
- Hyperparameter selection uses a fixed **stratified 12,000-sample training subset** and **3,000-sample validation subset**.
- Each of the three configurations is trained and evaluated on that fixed tuning subset.
- The best validation Macro-F1 configuration is then trained on the **full training split**.
- The final BERT metrics are computed once on the untouched full test split.

In [ ]:
tf.keras.backend.clear_session()

BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LENGTH = 64
BERT_TUNING_TRAIN_SIZE = min(12000, len(train_df) - 1)
BERT_TUNING_VALIDATION_SIZE = min(3000, len(validation_df) - 1)

bert_tune_train_df, _ = train_test_split(
    train_df[["tweet_text", "label"]],
    train_size=BERT_TUNING_TRAIN_SIZE,
    random_state=RANDOM_STATE,
    stratify=train_df["label"]
)

bert_tune_validation_df, _ = train_test_split(
    validation_df[["tweet_text", "label"]],
    train_size=BERT_TUNING_VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=validation_df["label"]
)

print("BERT tuning train size:", len(bert_tune_train_df))
print("BERT tuning validation size:", len(bert_tune_validation_df))

BERT tuning train size: 12000
BERT tuning validation size: 3000


In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)


def tokenize_bert(batch):
    return bert_tokenizer(
        batch["tweet_text"],
        truncation=True,
        max_length=BERT_MAX_LENGTH
    )


def dataframe_to_bert_dataset(frame):
    dataset = Dataset.from_pandas(frame[["tweet_text", "label"]].reset_index(drop=True))
    dataset = dataset.map(tokenize_bert, batched=True)
    dataset = dataset.rename_column("label", "labels")
    return dataset


bert_tune_train_dataset = dataframe_to_bert_dataset(bert_tune_train_df)
bert_tune_validation_dataset = dataframe_to_bert_dataset(bert_tune_validation_df)
bert_data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

print("BERT tokenization completed for the tuning subsets.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

BERT tokenization completed for the tuning subsets.


### 23.1 BERT Manual Hyperparameter Tuning

The configurations vary learning rate, batch size and epoch count. Validation Macro-F1 is the selection criterion, consistent with the multi-class evaluation used for the other models.

In [ ]:
bert_configs = [
    {"learning_rate": 2e-5, "batch_size": 16, "epochs": 1},
    {"learning_rate": 3e-5, "batch_size": 16, "epochs": 2},
    {"learning_rate": 2e-5, "batch_size": 32, "epochs": 2}
]

best_bert_config = None
best_bert_validation_f1 = -1

for config_id, config in enumerate(bert_configs, start=1):
    bert_model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME,
        num_labels=len(label_encoder.classes_)
    )

    training_arguments = TrainingArguments(
        output_dir=f"/content/bert_tuning_config_{config_id}",
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["batch_size"],
        learning_rate=config["learning_rate"],
        logging_steps=50,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=bert_model,
        args=training_arguments,
        train_dataset=bert_tune_train_dataset,
        eval_dataset=bert_tune_validation_dataset,
        data_collator=bert_data_collator
    )

    trainer.train()
    validation_output = trainer.predict(bert_tune_validation_dataset)
    validation_prediction = np.argmax(validation_output.predictions, axis=1)
    validation_true = np.array(bert_tune_validation_dataset["labels"])

    validation_accuracy = accuracy_score(validation_true, validation_prediction)
    validation_f1 = f1_score(validation_true, validation_prediction, average="macro")

    record_tuning("BERT Base", "BERT contextual representation", config_id, config, validation_accuracy, validation_f1)
    print("Config", config_id, config, "Validation Macro F1 =", round(validation_f1, 4))

    if validation_f1 > best_bert_validation_f1:
        best_bert_validation_f1 = validation_f1
        best_bert_config = config

    del trainer
    del bert_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

best_hyperparameters["BERT Base"] = best_bert_config
print("Best BERT Base config:", best_bert_config)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,1.626725
100,0.965242
150,0.682193
200,0.519070
250,0.462818
300,0.448546
350,0.410242
400,0.381370
450,0.378853
500,0.347619


...

Config 2 {'learning_rate': 3e-05, 'batch_size': 16, 'epochs': 2} Validation Macro F1 = 0.8704


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,1.459530
100,0.815710
150,0.529619
200,0.435726
250,0.399541
300,0.405894
350,0.398972
400,0.352857
450,0.290302
500,0.312635


Config 3 {'learning_rate': 2e-05, 'batch_size': 32, 'epochs': 2} Validation Macro F1 = 0.8588
Best BERT Base config: {'learning_rate': 3e-05, 'batch_size': 16, 'epochs': 2}


### 23.2 Final BERT Training on the Full Training Split

Only the best validation-selected BERT configuration is trained on the full training split. The test split remains untouched until the final prediction stage below.

In [ ]:
bert_full_train_dataset = dataframe_to_bert_dataset(train_df[["tweet_text", "label"]])
bert_test_dataset = dataframe_to_bert_dataset(test_df[["tweet_text", "label"]])

final_bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    id2label={index: label for index, label in enumerate(label_encoder.classes_)},
    label2id={label: index for index, label in enumerate(label_encoder.classes_)}
)

final_bert_arguments = TrainingArguments(
    output_dir="/content/bert_final",
    num_train_epochs=best_bert_config["epochs"],
    per_device_train_batch_size=best_bert_config["batch_size"],
    learning_rate=best_bert_config["learning_rate"],
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

final_bert_trainer = Trainer(
    model=final_bert_model,
    args=final_bert_arguments,
    train_dataset=bert_full_train_dataset,
    data_collator=bert_data_collator
)

final_bert_trainer.train()
print("Final BERT training completed.")

Map:   0%|          | 0/31064 [00:00<?, ? examples/s]

Map:   0%|          | 0/6657 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,1.435433
100,0.766759
150,0.588329
200,0.452384
250,0.420704
300,0.387408
350,0.429126
400,0.402586
450,0.391212
500,0.458622


Final BERT training completed.


In [ ]:
bert_test_output = final_bert_trainer.predict(bert_test_dataset)
bert_test_prediction = np.argmax(bert_test_output.predictions, axis=1)
bert_test_true = np.array(bert_test_dataset["labels"])

evaluate_predictions(
    "BERT Base",
    "BERT contextual representation",
    bert_test_true,
    bert_test_prediction
)

BERT Base
Representation: BERT contextual representation
Test Accuracy: 0.9064
Test Macro F1: 0.897

Full Classification Report:

                     precision    recall  f1-score   support

                age       0.98      0.98      0.98      1199
          ethnicity       0.98      0.98      0.98      1193
             gender       0.92      0.91      0.92      1166
  not_cyberbullying       0.75      0.73      0.74       964
other_cyberbullying       0.79      0.81      0.80       937
           religion       0.95      0.97      0.96      1198

           accuracy                           0.91      6657
          macro avg       0.90      0.90      0.90      6657
       weighted avg       0.91      0.91      0.91      6657

Confusion Matrix:
 [[1174    2    0   17    6    0]
 [   4 1168    3    6    8    4]
 [   0    2 1065   50   46    3]
 [   8   11   54  705  140   46]
 [   6    5   33  135  755    3]
 [   2    0    5   21    3 1167]]


(0.9064143007360673, 0.8969693595329166)

# Part D — Consolidated Final Results

The following comparison reproduces the final test metrics from the completed team experiment.
It allows this BERT/final-analysis notebook to remain independently readable without retraining every
classical and recurrent model inside the same runtime.


In [ ]:
team_results = [
    {"Model": "BERT Base", "Representation": "BERT contextual representation", "Accuracy": 0.906414, "Macro F1": 0.896969},
    {"Model": "Logistic Regression", "Representation": "TF-IDF", "Accuracy": 0.879075, "Macro F1": 0.868256},
    {"Model": "Random Forest", "Representation": "TF-IDF", "Accuracy": 0.877723, "Macro F1": 0.865386},
    {"Model": "Bidirectional GRU", "Representation": "Word2Vec", "Accuracy": 0.829953, "Macro F1": 0.813393},
    {"Model": "Bidirectional LSTM", "Representation": "Word2Vec", "Accuracy": 0.825597, "Macro F1": 0.812113},
    {"Model": "Naive Bayes", "Representation": "TF-IDF", "Accuracy": 0.815232, "Macro F1": 0.793034},
    {"Model": "LSTM", "Representation": "Word2Vec", "Accuracy": 0.803515, "Macro F1": 0.784541},
    {"Model": "GRU", "Representation": "Word2Vec", "Accuracy": 0.802313, "Macro F1": 0.781683},
    {"Model": "Bidirectional SimpleRNN", "Representation": "Word2Vec", "Accuracy": 0.755596, "Macro F1": 0.742255},
    {"Model": "SimpleRNN", "Representation": "Word2Vec", "Accuracy": 0.642181, "Macro F1": 0.606922},
]

results_df = pd.DataFrame(team_results).sort_values("Macro F1", ascending=False).reset_index(drop=True)
results_df


In [ ]:
plot_df = results_df.set_index("Model")[["Accuracy", "Macro F1"]]
plot_df.plot(kind="bar", figsize=(13, 6))
plt.title("Final Test Performance Across All Required Models")
plt.ylabel("Score")
plt.ylim(0, 1.0)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Best and Worst Performing Models

The final comparison is ranked by **Macro-F1**, which is appropriate for evaluating performance
across all six cyberbullying classes rather than relying only on overall accuracy.


In [ ]:
best_model_row = results_df.iloc[0]
worst_model_row = results_df.iloc[-1]

print("BEST MODEL")
print("Model:", best_model_row["Model"])
print("Representation:", best_model_row["Representation"])
print("Accuracy:", round(best_model_row["Accuracy"], 4))
print("Macro F1:", round(best_model_row["Macro F1"], 4))

print("\nWORST MODEL")
print("Model:", worst_model_row["Model"])
print("Representation:", worst_model_row["Representation"])
print("Accuracy:", round(worst_model_row["Accuracy"], 4))
print("Macro F1:", round(worst_model_row["Macro F1"], 4))


## BERT Confusion Matrix

Because this notebook independently trains and evaluates BERT, its confusion matrix is generated directly
from the BERT test predictions.


In [ ]:
bert_matrix = confusion_matrix(bert_test_true, bert_test_prediction)

plt.figure(figsize=(7, 6))
plt.imshow(bert_matrix, interpolation="nearest")
plt.title("BERT Base — Confusion Matrix")
plt.colorbar()

tick_positions = np.arange(len(label_encoder.classes_))
plt.xticks(tick_positions, label_encoder.classes_, rotation=45, ha="right")
plt.yticks(tick_positions, label_encoder.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

for row in range(bert_matrix.shape[0]):
    for column in range(bert_matrix.shape[1]):
        plt.text(column, row, str(bert_matrix[row, column]), ha="center", va="center")

plt.tight_layout()
plt.show()


## Final Interpretation

The completed experiment identifies **BERT Base** as the strongest model, with approximately
**90.64% test accuracy** and **0.8970 Macro-F1**. This makes the BERT section the most suitable
foundation for the final review-classification interface and deployment stage.


In [ ]:
results_df.to_csv("/content/model_comparison_results.csv", index=False)
print("Saved: /content/model_comparison_results.csv")
